In [ ]:
import json

with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate = json.load(f)

with open(
    "../data/interview_blueprint.json",
    "r",
    encoding="utf-8"
) as f:
    blueprint = json.load(f)

In [ ]:
current_question = {
    "question": (
        "You used FAISS in SceneSense AI. "
        "Why did you choose FAISS for semantic search?"
    ),
    "topic": "Vector Search",
    "category": "technical_project",
    "difficulty": "medium",
    "question_type": "technical_project",
    "expected_concepts": [
        "vector embeddings",
        "similarity search",
        "efficient retrieval",
        "FAISS indexing"
    ]
}

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class AnswerEvaluation(BaseModel):

    relevance: float = Field(
        ge=0,
        le=10
    )

    technical_accuracy: float = Field(
        ge=0,
        le=10
    )

    depth: float = Field(
        ge=0,
        le=10
    )

    reasoning: float = Field(
        ge=0,
        le=10
    )

    clarity: float = Field(
        ge=0,
        le=10
    )

    confidence: float = Field(
        ge=0,
        le=10
    )

    overall_score: float = Field(
        ge=0,
        le=10
    )

    strengths: list[str] = Field(
        default_factory=list
    )

    weaknesses: list[str] = Field(
        default_factory=list
    )

    missing_concepts: list[str] = Field(
        default_factory=list
    )

    suggested_follow_up: str | None = None

    should_challenge: bool = False

    reasoning_summary: str = ""

In [ ]:
EVALUATOR_PROMPT = """
You are the Answer Evaluation Agent for InterviewHive.

You are an expert technical interviewer evaluating a candidate's
answer to an interview question.

Your job is to evaluate ONLY the candidate's answer against:
1. The interview question
2. The expected concepts
3. The candidate's provided context when relevant

Do NOT evaluate the candidate as a person.

EVALUATION RULES

1. Evaluate the actual answer, not what you think the candidate meant.

2. Do not invent information that the candidate did not provide.

3. Technical correctness is more important than verbosity.

4. A short but technically correct answer can receive a high score.

5. A long answer containing incorrect information must be penalized.

6. Identify missing concepts only when those concepts are relevant
   to the question.

7. Distinguish between:
   - incorrect information
   - missing information
   - information that was simply not discussed

8. Relevance measures whether the answer actually addresses the
   question.

9. Technical accuracy measures whether the technical claims are
   correct.

10. Depth measures how thoroughly the candidate explains the topic.

11. Reasoning measures the candidate's ability to explain WHY,
    HOW, trade-offs, decisions, or implications.

12. Clarity measures how understandable and logically organized
    the answer is.

13. Confidence measures how confidently and appropriately the
    candidate communicates their answer. Do not assume confidence
    merely from writing style.

SCORING

Score every dimension from 0 to 10.

0-2  = very poor
3-4  = weak
5-6  = acceptable
7-8  = strong
9-10 = excellent

Use the full range when appropriate.

OVERALL SCORE

The overall_score should represent the quality of the complete answer.

Consider all evaluation dimensions, with technical accuracy and
relevance being especially important.

STRENGTHS

List the specific things the candidate did well.

Do not invent strengths.

WEAKNESSES

List specific problems or areas where the answer could be improved.

MISSING CONCEPTS

Only include concepts from expected_concepts that are genuinely
missing or insufficiently addressed.

Do not penalize the candidate for concepts that were not required.

FOLLOW-UP

If a meaningful follow-up question would help test the candidate's
understanding, provide one.

The follow-up should target the most important weakness or uncertainty.

If no follow-up is necessary, set suggested_follow_up to null.

CHALLENGE DECISION

Set should_challenge to true when:
- the candidate gives an incorrect technical claim,
- the answer is too vague to establish understanding,
- the candidate makes a significant unsupported claim,
- or deeper questioning would meaningfully test their understanding.

Otherwise set it to false.

OUTPUT FORMAT

Return ONLY a valid JSON object.

The output MUST EXACTLY match the provided AnswerEvaluation schema.

Use EXACTLY these field names:

- relevance
- technical_accuracy
- depth
- reasoning
- clarity
- confidence
- overall_score
- strengths
- weaknesses
- missing_concepts
- suggested_follow_up
- should_challenge
- reasoning_summary

DO NOT use alternative field names such as:

- score
- feedback
- follow_up
- accuracy
- explanation

For example, DO NOT return:

{
    "score": 9,
    "feedback": "...",
    "follow_up": null
}

Instead return the complete AnswerEvaluation structure.

Every required field MUST be present.

Return ONLY JSON. No Markdown. No explanation outside the JSON.
"""

In [ ]:
evaluation_input = {
    "target_role": blueprint["target_role"],
    "question": current_question,
    "candidate_answer": candidate_answer
}

evaluation_text = json.dumps(
    evaluation_input,
    indent=2,
    ensure_ascii=False
)

print(evaluation_text)

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [ ]:
schema = AnswerEvaluation.model_json_schema()

In [ ]:
evaluation_tests = [
    {
        "name": "strong_answer",
        "answer": """
        FAISS is a library for efficient similarity search over
        dense vectors. In SceneSense AI, I converted movie
        descriptions and queries into embeddings and used FAISS
        to efficiently retrieve similar vectors instead of
        comparing every item manually.
        """,
        "expected": "strong"
    },

    {
        "name": "vague_answer",
        "answer": """
        I used FAISS because it is fast and efficient and it
        worked well for my project.
        """,
        "expected": "medium"
    },

    {
        "name": "incorrect_answer",
        "answer": """
        FAISS is a classification algorithm that predicts which
        movie a user will like based on training labels.
        """,
        "expected": "weak"
    },

    {
        "name": "irrelevant_answer",
        "answer": """
        Python is a programming language and I have used it
        in many projects.
        """,
        "expected": "weak"
    }
]

In [ ]:
def evaluate_answer(
    question,
    candidate_answer
):
    
    evaluation_input = {
        "target_role": blueprint["target_role"],
        "question": question,
        "candidate_answer": candidate_answer
    }

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": EVALUATOR_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(
                    evaluation_input,
                    indent=2,
                    ensure_ascii=False
                )
            }
        ],
        temperature=0
    )

    response_text = response.choices[0].message.content

    parsed = json.loads(response_text)

    return AnswerEvaluation.model_validate(
        parsed
    )

In [ ]:
test_results = []

for test in evaluation_tests:

    result = evaluate_answer(
        current_question,
        test["answer"]
    )

    test_results.append({
        "name": test["name"],
        "expected": test["expected"],
        "score": result.overall_score,
        "technical_accuracy": result.technical_accuracy,
        "depth": result.depth,
        "reasoning": result.reasoning
    })

    print("\n", test["name"])
    print("Expected:", test["expected"])
    print("Score:", result.overall_score)

In [ ]:
def score_category(score):

    if score >= 8:
        return "strong"

    elif score >= 5:
        return "medium"

    else:
        return "weak"
    
for result in test_results:

    predicted = score_category(
        result["score"]
    )

    print(
        result["name"],
        "| expected:",
        result["expected"],
        "| predicted:",
        predicted
    )
    


In [ ]:
def determine_answer_state(evaluation):

    if evaluation.technical_accuracy < 5:
        return "technical_gap"

    if evaluation.depth < 5:
        return "shallow"

    if evaluation.reasoning < 5:
        return "weak_reasoning"

    if evaluation.overall_score >= 8:
        return "strong"

    return "acceptable"

In [ ]:
result = evaluate_answer(
    current_question,
    evaluation_tests[1]["answer"]
)

print(
    determine_answer_state(result)
)

In [ ]:
def create_evaluation_signal(evaluation):

    return {
        "overall_score": evaluation.overall_score,
        "technical_accuracy": evaluation.technical_accuracy,
        "depth": evaluation.depth,
        "reasoning": evaluation.reasoning,
        "state": determine_answer_state(evaluation),
        "should_challenge": evaluation.should_challenge,
        "suggested_follow_up": evaluation.suggested_follow_up,
        "missing_concepts": evaluation.missing_concepts
    }

In [ ]:
signal = create_evaluation_signal(result)

print(json.dumps(
    signal,
    indent=2
))

In [197]:
with open(
    "../data/evaluation_test_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_results,
        f,
        indent=2,
        ensure_ascii=False
    )